# BeaverTails Expert Routing Analysis

Analysis of expert routing patterns during **response generation** (refusal vs. harmful responses).

Based on Fayyaz et al. (2025) matched pairs approach.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter
from pathlib import Path

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

## 1. Load Data

In [ ]:
# Load the routing data
data_file = "expert_routing_analysis/beavertails_routing_data.json"

with open(data_file, 'r') as f:
    data = json.load(f)

refusal_results = data["refusal_results"]
harmful_results = data["harmful_results"]

print(f"Model: {data['model']}")
print(f"Dataset: {data['dataset']} ({data.get('beavertails_split', 'unknown')})")
print(f"Number of layers: {data['num_layers']}")
print(f"Experts per layer: {data['num_experts']}")
print(f"Active experts per token: {data['experts_per_token']}")
print(f"\nRefusal responses: {len(refusal_results)}")
print(f"Harmful responses: {len(harmful_results)}")
print(f"Max response tokens: {data['max_response_tokens']}")

## 2. Expert Usage Distribution

How often is each expert used in refusal vs. harmful responses?

In [ ]:
def get_expert_frequencies(results, label):
    """Calculate expert usage frequencies for each layer."""
    num_layers = len(results[0]["layer_routing"])
    num_experts = data['num_experts']
    
    frequencies = np.zeros((num_layers, num_experts))
    
    for layer_idx in range(num_layers):
        layer_key = f"layer_{layer_idx}"
        all_experts = []
        
        for result in results:
            if layer_key in result["layer_routing"]:
                top_experts = result["layer_routing"][layer_key]["top_expert"]
                all_experts.extend(top_experts)
        
        if all_experts:
            expert_counts = Counter(all_experts)
            total = len(all_experts)
            
            for expert_id in range(num_experts):
                frequencies[layer_idx, expert_id] = expert_counts.get(expert_id, 0) / total
    
    return frequencies

refusal_freq = get_expert_frequencies(refusal_results, "refusal")
harmful_freq = get_expert_frequencies(harmful_results, "harmful")

In [ ]:
# Visualize expert usage distributions for a few layers
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

# Show layers with highest variance
layer_variance = np.var(refusal_freq - harmful_freq, axis=1)
top_layers = np.argsort(layer_variance)[-6:][::-1]

for idx, layer_idx in enumerate(top_layers):
    ax = axes[idx]
    
    x = np.arange(data['num_experts'])
    width = 0.35
    
    ax.bar(x - width/2, refusal_freq[layer_idx], width, label='Refusal', alpha=0.8, color='blue')
    ax.bar(x + width/2, harmful_freq[layer_idx], width, label='Harmful', alpha=0.8, color='red')
    
    ax.set_xlabel('Expert ID')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Layer {layer_idx} Expert Usage')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('expert_routing_analysis/expert_usage_by_layer.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Showing layers with highest variance in expert usage: {top_layers}")

## 3. Expert Preference Analysis

Which experts are preferentially used in refusal vs. harmful responses?

In [ ]:
# Calculate differences (refusal - harmful)
expert_differences = refusal_freq - harmful_freq

# Create heatmap
fig, ax = plt.subplots(figsize=(16, 8))

sns.heatmap(
    expert_differences,
    cmap='RdBu_r',
    center=0,
    vmin=-0.15,
    vmax=0.15,
    cbar_kws={'label': 'Frequency Difference (Refusal - Harmful)'},
    ax=ax
)

ax.set_xlabel('Expert ID')
ax.set_ylabel('Layer')
ax.set_title('Expert Routing Preferences During Response Generation\n(Blue = Refusal-Preferred, Red = Harmful-Preferred)')

plt.tight_layout()
plt.savefig('expert_routing_analysis/expert_preferences_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Top Refusal-Associated and Harmful-Associated Experts

Identify the most discriminative experts for intervention.

In [ ]:
# Find top experts per layer
top_refusal_experts = {}
top_harmful_experts = {}

for layer_idx in range(data['num_layers']):
    diffs = expert_differences[layer_idx]
    
    # Top refusal-preferred (positive difference)
    refusal_mask = diffs > 0.01  # At least 1% difference
    if np.any(refusal_mask):
        refusal_experts = np.where(refusal_mask)[0]
        refusal_values = diffs[refusal_mask]
        top_idx = np.argsort(refusal_values)[-3:][::-1]  # Top 3
        top_refusal_experts[layer_idx] = list(zip(
            refusal_experts[top_idx],
            refusal_values[top_idx]
        ))
    
    # Top harmful-preferred (negative difference)
    harmful_mask = diffs < -0.01
    if np.any(harmful_mask):
        harmful_experts = np.where(harmful_mask)[0]
        harmful_values = diffs[harmful_mask]
        top_idx = np.argsort(harmful_values)[:3]  # Top 3 (most negative)
        top_harmful_experts[layer_idx] = list(zip(
            harmful_experts[top_idx],
            harmful_values[top_idx]
        ))

print("TOP REFUSAL-PREFERRED EXPERTS (during response generation)")
print("="*70)
for layer_idx in sorted(top_refusal_experts.keys()):
    print(f"\nLayer {layer_idx}:")
    for expert_id, diff in top_refusal_experts[layer_idx]:
        refusal_pct = refusal_freq[layer_idx, expert_id] * 100
        harmful_pct = harmful_freq[layer_idx, expert_id] * 100
        print(f"  Expert {expert_id:2d}: +{diff:.4f} (refusal: {refusal_pct:5.2f}%, harmful: {harmful_pct:5.2f}%)")

print("\n" + "="*70)
print("TOP HARMFUL-PREFERRED EXPERTS (during response generation)")
print("="*70)
for layer_idx in sorted(top_harmful_experts.keys()):
    print(f"\nLayer {layer_idx}:")
    for expert_id, diff in top_harmful_experts[layer_idx]:
        refusal_pct = refusal_freq[layer_idx, expert_id] * 100
        harmful_pct = harmful_freq[layer_idx, expert_id] * 100
        print(f"  Expert {expert_id:2d}: {diff:.4f} (refusal: {refusal_pct:5.2f}%, harmful: {harmful_pct:5.2f}%)")

## 5. Visualize Top Discriminative Experts

In [ ]:
# Create a visualization of top discriminative experts
# Flatten all expert differences and find the most extreme ones

all_expert_diffs = []
for layer_idx in range(data['num_layers']):
    for expert_id in range(data['num_experts']):
        diff = expert_differences[layer_idx, expert_id]
        if abs(diff) > 0.01:  # At least 1% difference
            all_expert_diffs.append({
                'layer': layer_idx,
                'expert': expert_id,
                'difference': diff,
                'refusal_freq': refusal_freq[layer_idx, expert_id],
                'harmful_freq': harmful_freq[layer_idx, expert_id],
                'label': f"L{layer_idx}E{expert_id}"
            })

# Sort by absolute difference
all_expert_diffs.sort(key=lambda x: abs(x['difference']), reverse=True)

# Take top 20
top_20 = all_expert_diffs[:20]

# Plot
fig, ax = plt.subplots(figsize=(14, 8))

labels = [e['label'] for e in top_20]
diffs = [e['difference'] for e in top_20]
colors = ['blue' if d > 0 else 'red' for d in diffs]

y_pos = np.arange(len(labels))
ax.barh(y_pos, diffs, color=colors, alpha=0.7)

ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set_xlabel('Frequency Difference (Refusal - Harmful)')
ax.set_title('Top 20 Most Discriminative Experts During Response Generation')
ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax.grid(alpha=0.3, axis='x')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='blue', alpha=0.7, label='Refusal-Preferred'),
    Patch(facecolor='red', alpha=0.7, label='Harmful-Preferred')
]
ax.legend(handles=legend_elements, loc='best')

plt.tight_layout()
plt.savefig('expert_routing_analysis/top_discriminative_experts.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Token Position Analysis

Do experts change across the response? (Early vs. late tokens)

In [ ]:
def analyze_position_patterns(results, layer_idx=10):
    """Analyze expert usage by position in response."""
    layer_key = f"layer_{layer_idx}"
    
    early_experts = []
    middle_experts = []
    late_experts = []
    
    for result in results:
        if layer_key not in result["layer_routing"]:
            continue
        
        top_experts = result["layer_routing"][layer_key]["top_expert"]
        response_len = len(top_experts)
        
        if response_len < 3:
            continue
        
        # Divide into thirds
        early_end = response_len // 3
        middle_end = 2 * response_len // 3
        
        early_experts.extend(top_experts[:early_end])
        middle_experts.extend(top_experts[early_end:middle_end])
        late_experts.extend(top_experts[middle_end:])
    
    # Get frequencies
    num_experts = data['num_experts']
    early_freq = np.array([Counter(early_experts).get(i, 0) / (len(early_experts) + 1e-10) for i in range(num_experts)])
    middle_freq = np.array([Counter(middle_experts).get(i, 0) / (len(middle_experts) + 1e-10) for i in range(num_experts)])
    late_freq = np.array([Counter(late_experts).get(i, 0) / (len(late_experts) + 1e-10) for i in range(num_experts)])
    
    return early_freq, middle_freq, late_freq

# Analyze layer 10 (often shows interesting patterns)
refusal_early, refusal_middle, refusal_late = analyze_position_patterns(refusal_results, layer_idx=10)
harmful_early, harmful_middle, harmful_late = analyze_position_patterns(harmful_results, layer_idx=10)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Refusal responses
ax = axes[0]
x = np.arange(data['num_experts'])
width = 0.25

ax.bar(x - width, refusal_early, width, label='Early', alpha=0.7)
ax.bar(x, refusal_middle, width, label='Middle', alpha=0.7)
ax.bar(x + width, refusal_late, width, label='Late', alpha=0.7)

ax.set_xlabel('Expert ID')
ax.set_ylabel('Frequency')
ax.set_title('Layer 10: Expert Usage by Position (REFUSAL)')
ax.legend()
ax.grid(alpha=0.3)

# Harmful responses
ax = axes[1]
ax.bar(x - width, harmful_early, width, label='Early', alpha=0.7)
ax.bar(x, harmful_middle, width, label='Middle', alpha=0.7)
ax.bar(x + width, harmful_late, width, label='Late', alpha=0.7)

ax.set_xlabel('Expert ID')
ax.set_ylabel('Frequency')
ax.set_title('Layer 10: Expert Usage by Position (HARMFUL)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('expert_routing_analysis/position_analysis_layer10.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Intervention Recommendations

Based on the analysis, which paired interventions should we test?

In [ ]:
print("INTERVENTION RECOMMENDATIONS")
print("="*70)
print("\nBased on response-level expert routing analysis:")
print("\nTop 5 layers for intervention:\n")

# Find layers with highest discriminative power
layer_discriminative_power = np.max(np.abs(expert_differences), axis=1)
top_layers = np.argsort(layer_discriminative_power)[-5:][::-1]

intervention_configs = []

for rank, layer_idx in enumerate(top_layers, 1):
    # Get top refusal and harmful expert for this layer
    diffs = expert_differences[layer_idx]
    
    refusal_expert = np.argmax(diffs)
    harmful_expert = np.argmin(diffs)
    
    refusal_diff = diffs[refusal_expert]
    harmful_diff = diffs[harmful_expert]
    
    print(f"{rank}. Layer {layer_idx}:")
    print(f"   Most refusal-preferred:  Expert {refusal_expert} (+{refusal_diff:.4f})")
    print(f"   Most harmful-preferred:  Expert {harmful_expert} ({harmful_diff:.4f})")
    print(f"   ")
    print(f"   Refusal Induction:  Force E{refusal_expert}, Suppress E{harmful_expert}")
    print(f"   Response Induction: Force E{harmful_expert}, Suppress E{refusal_expert}")
    print()
    
    intervention_configs.append({
        'layer': layer_idx,
        'refusal_expert': int(refusal_expert),
        'harmful_expert': int(harmful_expert),
        'refusal_diff': float(refusal_diff),
        'harmful_diff': float(harmful_diff)
    })

# Save intervention recommendations
with open('expert_routing_analysis/intervention_recommendations.json', 'w') as f:
    json.dump({
        'top_layers': intervention_configs,
        'all_expert_differences': expert_differences.tolist()
    }, f, indent=2)

print("\nIntervention recommendations saved to:")
print("  expert_routing_analysis/intervention_recommendations.json")

## 8. Summary Statistics

In [ ]:
# Create summary dataframe
summary_data = []

for layer_idx in range(data['num_layers']):
    max_diff = np.max(expert_differences[layer_idx])
    min_diff = np.min(expert_differences[layer_idx])
    mean_abs_diff = np.mean(np.abs(expert_differences[layer_idx]))
    
    summary_data.append({
        'Layer': layer_idx,
        'Max Refusal Preference': max_diff,
        'Max Harmful Preference': abs(min_diff),
        'Mean Abs Difference': mean_abs_diff,
        'Discriminative Power': max(max_diff, abs(min_diff))
    })

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values('Discriminative Power', ascending=False)

print("LAYER DISCRIMINATIVE POWER")
print("="*70)
print(summary_df.to_string(index=False))

# Save summary
summary_df.to_csv('expert_routing_analysis/layer_summary.csv', index=False)
print("\nSummary saved to: expert_routing_analysis/layer_summary.csv")

## 9. Export Expert Differences for Intervention Code

In [ ]:
# Export in format suitable for expert_intervention_hooks_v3.py
expert_diffs_for_export = {}

for layer_idx in range(data['num_layers']):
    expert_diffs_for_export[str(layer_idx)] = [
        [int(expert_id), float(diff)]
        for expert_id, diff in enumerate(expert_differences[layer_idx])
    ]

with open('expert_explore/beavertails_expert_diffs.json', 'w') as f:
    json.dump(expert_diffs_for_export, f, indent=2)

print("Expert differences exported to: expert_explore/beavertails_expert_diffs.json")
print("This can be used to create threshold-based interventions.")